# SAE Decoder PCA / t-SNE Visualization

这个 notebook 只保留与 PCA / t-SNE 相关的代码：读取 `pipeline_summary.json`，加载 `standard` 和 `attnres` 的 SAE decoder `W_dec`，并做二维投影可视化。

## 1. 定位结果目录

先自动定位仓库根目录和 `pipeline_summary.json`。后面的 PCA / t-SNE 都依赖这里找到的结果路径。

In [ ]:
from __future__ import annotations

import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch

plt.rcParams["axes.unicode_minus"] = False
MODEL_COLORS = {
    "standard": "#1f77b4",
    "attnres": "#ff7f0e",
}


def find_repo_root(start: Path | None = None) -> Path:
    candidate = Path.cwd() if start is None else start
    for path in [candidate, *candidate.parents]:
        if (path / "stream_analysis").exists() and (path / "scripts").exists():
            return path
    raise FileNotFoundError("Could not locate the repo root from the current working directory.")


REPO_ROOT = find_repo_root()
SUMMARY_PATH = REPO_ROOT / "outputs" / "sae_dual_pipeline" / "pipeline_summary.json"
REPO_ROOT, SUMMARY_PATH


## 2. 加载 SAE checkpoint

这里只读取 PCA / t-SNE 需要的最小信息：每个模型对应的 `sae_best_checkpoint`。

In [ ]:
if not SUMMARY_PATH.is_file():
    raise FileNotFoundError(
        f"Missing pipeline summary: {SUMMARY_PATH}\n"
        "先运行 accelerate launch scripts/run_sae_pipeline.py ... 生成双模型 SAE 结果。"
    )

with SUMMARY_PATH.open("r", encoding="utf-8") as handle:
    pipeline_summary = json.load(handle)

model_order = list(pipeline_summary["models"].keys())
checkpoint_map = {
    model_type: Path(payload["sae_best_checkpoint"])
    for model_type, payload in pipeline_summary["models"].items()
}

for model_type, checkpoint_path in checkpoint_map.items():
    if not checkpoint_path.is_file():
        raise FileNotFoundError(f"Missing SAE checkpoint for {model_type}: {checkpoint_path}")

checkpoint_map


## 3. Decoder Atom PCA / t-SNE

这里直接对归一化后的 decoder 原子 `W_dec` 做二维投影：

- `PCA` 用来看全局线性主方向
- `t-SNE` 用来看局部邻域结构

为了稳定 t-SNE，先把 decoder 原子预降到 `50` 维，再投影到二维。下面按投影方法分组，把 `standard` 和 `attnres` 并列显示，便于横向比较。

In [ ]:
try:
    from sklearn.decomposition import PCA
    from sklearn.manifold import TSNE
except Exception as exc:
    raise ImportError("This section requires scikit-learn for PCA / t-SNE.") from exc

DECODER_PROJECTION_MAX_LATENTS = None
DECODER_TSNE_PRE_PCA_DIM = 50
DECODER_TSNE_PERPLEXITY = 35
DECODER_TSNE_RANDOM_STATE = 42
DECODER_TSNE_METRIC = "cosine"
DECODER_PROJECTION_POINT_SIZE = 10
DECODER_PROJECTION_ALPHA = 0.5


def load_decoder_atom_matrix(checkpoint_path: str | Path, normalize: bool = True) -> np.ndarray:
    payload = torch.load(Path(checkpoint_path), map_location="cpu", weights_only=False)
    w_dec = payload["model_state"]["W_dec"].detach().to(dtype=torch.float32).cpu().numpy()
    if normalize:
        norms = np.linalg.norm(w_dec, axis=1, keepdims=True)
        w_dec = w_dec / np.clip(norms, 1e-8, None)
    return np.asarray(w_dec, dtype=np.float32)


def maybe_subsample_decoder_atoms(atom_matrix: np.ndarray, max_latents: int | None = DECODER_PROJECTION_MAX_LATENTS):
    if max_latents is None or atom_matrix.shape[0] <= max_latents:
        indices = np.arange(atom_matrix.shape[0], dtype=int)
        return atom_matrix, indices
    indices = np.linspace(0, atom_matrix.shape[0] - 1, num=max_latents, dtype=int)
    return atom_matrix[indices], indices


def compute_decoder_projection_bundle(
    atom_matrix: np.ndarray,
    *,
    max_latents: int | None = DECODER_PROJECTION_MAX_LATENTS,
    tsne_pre_pca_dim: int = DECODER_TSNE_PRE_PCA_DIM,
    tsne_perplexity: float = DECODER_TSNE_PERPLEXITY,
    tsne_metric: str = DECODER_TSNE_METRIC,
    tsne_random_state: int = DECODER_TSNE_RANDOM_STATE,
):
    sampled_atoms, sampled_indices = maybe_subsample_decoder_atoms(atom_matrix, max_latents=max_latents)
    n_samples, n_features = sampled_atoms.shape
    if n_samples < 3:
        raise RuntimeError(f"Need at least 3 latents for projection, got {n_samples}.")

    pca_2d = PCA(n_components=2)
    pca_coords = pca_2d.fit_transform(sampled_atoms)
    pca_var = pca_2d.explained_variance_ratio_

    pre_pca_dim = min(tsne_pre_pca_dim, n_features, max(2, n_samples - 1))
    if pre_pca_dim >= 2 and pre_pca_dim < n_features:
        tsne_input = PCA(n_components=pre_pca_dim).fit_transform(sampled_atoms)
    else:
        tsne_input = sampled_atoms
        pre_pca_dim = tsne_input.shape[1]

    max_valid_perplexity = min(float(tsne_perplexity), float(n_samples - 1))
    if max_valid_perplexity >= n_samples:
        max_valid_perplexity = max(2.0, float(n_samples - 1) / 3.0)
    if max_valid_perplexity < 2.0:
        max_valid_perplexity = 2.0

    tsne = TSNE(
        n_components=2,
        perplexity=max_valid_perplexity,
        init="pca",
        learning_rate="auto",
        metric=tsne_metric,
        random_state=tsne_random_state,
    )
    tsne_coords = tsne.fit_transform(tsne_input)

    return {
        "sampled_indices": sampled_indices,
        "pca_coords": np.asarray(pca_coords, dtype=float),
        "pca_explained_variance_ratio": np.asarray(pca_var, dtype=float),
        "tsne_coords": np.asarray(tsne_coords, dtype=float),
        "tsne_pre_pca_dim": int(pre_pca_dim),
        "tsne_perplexity": float(max_valid_perplexity),
        "tsne_metric": tsne_metric,
        "n_samples": int(n_samples),
        "n_features": int(n_features),
    }


def plot_single_projection(axis, coords: np.ndarray, *, model_type: str, title: str, xlabel: str, ylabel: str):
    axis.scatter(
        coords[:, 0],
        coords[:, 1],
        s=DECODER_PROJECTION_POINT_SIZE,
        c=MODEL_COLORS.get(model_type, "#4c4c4c"),
        alpha=DECODER_PROJECTION_ALPHA,
        linewidths=0,
    )
    axis.set_title(title)
    axis.set_xlabel(xlabel)
    axis.set_ylabel(ylabel)
    axis.grid(alpha=0.2)


def print_projection_summary(model_type: str, atom_matrix: np.ndarray, bundle: dict[str, object]):
    print(f"[{model_type}] decoder atom projection summary")
    print(f"  normalized_atom_shape: {atom_matrix.shape}")
    print(f"  sampled_latent_count: {bundle['n_samples']}")
    print(f"  feature_dim: {bundle['n_features']}")
    print(
        "  PCA explained variance ratio: "
        f"[{bundle['pca_explained_variance_ratio'][0]:.4f}, {bundle['pca_explained_variance_ratio'][1]:.4f}]"
    )
    print(f"  t-SNE pre-PCA dim: {bundle['tsne_pre_pca_dim']}")
    print(f"  t-SNE perplexity: {bundle['tsne_perplexity']:.1f}")
    print(f"  t-SNE metric: {bundle['tsne_metric']}")


projection_payloads = {}
for model_type in model_order:
    atom_matrix = load_decoder_atom_matrix(checkpoint_map[model_type], normalize=True)
    bundle = compute_decoder_projection_bundle(atom_matrix)
    print_projection_summary(model_type, atom_matrix, bundle)
    projection_payloads[model_type] = {
        "atom_matrix": atom_matrix,
        "bundle": bundle,
    }

fig, axes = plt.subplots(2, len(model_order), figsize=(7 * len(model_order), 10), dpi=220, squeeze=False)
fig.suptitle("Decoder atom projection comparison", fontsize=14)

for col_idx, model_type in enumerate(model_order):
    bundle = projection_payloads[model_type]["bundle"]
    plot_single_projection(
        axes[0, col_idx],
        bundle["pca_coords"],
        model_type=model_type,
        title=f"{model_type}: decoder atoms PCA",
        xlabel="PC 1",
        ylabel="PC 2",
    )
    plot_single_projection(
        axes[1, col_idx],
        bundle["tsne_coords"],
        model_type=model_type,
        title=f"{model_type}: decoder atoms t-SNE",
        xlabel="t-SNE 1",
        ylabel="t-SNE 2",
    )

plt.tight_layout(rect=[0, 0.02, 1, 0.96])
plt.show()
